In [19]:
!pip install pymupdf

  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl (19.2 MB)


In [3]:
!pip install -U langchain langchain-core langchain-community langchain-pinecone pinecone-client

  Using cached langchain-1.3.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langgraph-1.2.1-py3-none-any.whl.metadata (8.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\thaku\\AppData\\Local\\Temp\\pip-unpack-tyz___56\\langchain-1.3.1-py3-none-any.whl'
Consider using the `--user` option or check the permissions.



In [15]:
# pip install langchain_pinecone sentence-transformers
!pip install -U langchain-pinecone

In [1]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
load_dotenv()

True

In [2]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
print(str(PINECONE_API_KEY))


pcsk_3snxbU_2qN4QnURwsXFcvg8YAaAJos2nwkAyEu9oiAXnmvbnopn7FMTr3kE1smeQnppN2R


In [3]:
pc = Pinecone(api_key = str(PINECONE_API_KEY))  # CONNECTION BUILD VIA API

In [4]:
index_name = 'genai'

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 1024,
        metric = 'cosine',
        spec = ServerlessSpec(cloud = 'aws', region = 'us-east-1')
    )

# Dense  : semantic search
# sparse : keyword handling

index = pc.Index(index_name)

1. I need an embedding model (embeddings : Huggingface / Ollama/ Pinecone)
2. Data -> vectors(Embeddings)
3. vectorstore -> 
4. Retriever

In [15]:
# from langchain_huggingface import HuggingFaceEmbeddings
# embedding_model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

In [16]:
# from langchain_ollama import OllamaEmbeddings
# embedding_model = OllamaEmbeddings(model = "bge-m3")

In [5]:
from langchain_pinecone import PineconeEmbeddings
embedding_model = PineconeEmbeddings()

In [6]:
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore(
    index = index,
    embedding = embedding_model
)

In [23]:
# !pip uninstall langchain-pinecone -y
# !pip install langchain-pinecone==0.2.13

In [7]:
vectorstore

In [12]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

vectorstore.add_documents(documents=documents, ids = [str(i) for i in range(len(documents))])

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [29]:
vectorstore.delete(ids = [str(i) for i in range(len(documents))])

In [32]:
from langchain_pinecone import PineconeRerank, PineconeSparseVectorStore

In [11]:
retriever = vectorstore.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k' : 10}
)

In [12]:
query = input("Enter your question : ")

In [13]:
retriever.invoke(query) # runnable


[Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Docu

In [ ]:
# pdf file splitting -> chunking -> embeddings -> embeddings save pinecone vector Database :- indexing(Table)


index_name = 'mlsystemdesign'

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 1024,
        metric = 'cosine',
        spec = ServerlessSpec(cloud = 'aws', region = 'us-east-1')
    )

# Dense  : semantic search
# sparse : keyword handling

index = pc.Index(index_name)


In [17]:
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader(file_path = r"D:\live_Projects\langchain\data\Machine learning System Design.pdf")

In [20]:
docs = loader.load()

In [29]:
type(docs[0])

langchain_core.documents.base.Document

In [26]:
docs[1].page_content

'Designing Machine Learning\nSystems\nAn Iterative Process for Production-Ready\nApplications\nWith Early Release ebooks, you get books in their earliest form—the\nauthor’s raw and unedited content as they write—so you can take\nadvantage of these technologies long before the official release of these\ntitles.\nChip Huyen'

In [44]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4500, 
    chunk_overlap = 200
)

In [45]:
chunks = splitter.split_documents(docs)

In [46]:
len(chunks)

318

In [47]:
chunks[1].page_content

'Designing Machine Learning Systems\nby Chip Huyen\nCopyright © 2022 Huyen Thi Khanh Nguyen. All rights reserved.\nPrinted in the United States of America.\nPublished by O’Reilly Media, Inc., 1005 Gravenstein Highway North,\nSebastopol, CA 95472.\nO’Reilly books may be purchased for educational, business, or sales\npromotional use. Online editions are also available for most titles\n(http://oreilly.com). For more information, contact our\ncorporate/institutional sales department: 800-998-9938 or\ncorporate@oreilly.com.\nAcquisitions Editor: Rebecca Novack\nDevelopment Editor: Jill Leonard\nProduction Editor: Kristen Brown\nCopyeditor:\nProofreader:\nIndexer:\nInterior Designer: David Futato\nCover Designer: Karen Montgomery\nIllustrator: Kate Dullea\nMarch 2022: First Edition\nRevision History for the Early Release'

In [48]:
from langchain_pinecone import PineconeVectorStore
ml_vector_store = PineconeVectorStore(
    index = index,
    embedding = embedding_model
)

In [49]:
ml_vector_store

In [51]:
ml_vector_store.add_documents(chunks, id = [str(i) for i in range(len(chunks))])

['8df75084-f05f-4103-b472-c9153290d8d0',
 'cb106f7c-7fd7-48d0-9f46-d2939da0d9cc',
 'ecf718fe-dfe5-430c-9d81-23047de1e121',
 '7a8cd1ee-1f13-4d99-9a8d-2d57b16b3258',
 '9d016f37-f23e-4989-93fa-7e848e62cbb2',
 '9b8b7a49-e571-489e-9ee2-8899a9936847',
 '483dc7b4-b6fd-4c72-a3f8-dae0bf197496',
 '2666b98b-fbc7-4346-b941-1f67ef20b8e4',
 'd4251d41-2372-4435-a0c4-e15cf5e59134',
 '85f22133-9c00-4696-9dc2-f1d767e4e4a6',
 '04e70565-0aa5-4aff-99e4-529f2d3467c9',
 '148d7bfb-2e08-43f2-9c0f-4fbdfc502585',
 '3bb18aa9-278b-4c5a-ac46-f5768c69c3cd',
 'ac22c8ab-c55a-4d1f-97c3-41613791a671',
 'd9c17508-dc7d-4c73-bf40-00b62f07da54',
 'fbe89a2d-4177-43ad-a5e5-081afadc1885',
 '2128b390-c404-48d4-ae3a-5b9f4b698dd2',
 '1d61c3c1-4e8b-4559-bd50-092fd63bc04f',
 '4b84a609-83de-487c-b09b-4715c06a07da',
 'fbd3303d-1873-463b-adcf-9f103c06f952',
 'd5e7b1f5-a5b1-4270-b3a3-247372a2053b',
 '492dac81-769a-4591-ae75-17e63ee0fb37',
 '6ec0898c-6245-4dbc-a60d-369bfce248dd',
 '251bccc5-ba31-44a9-8405-422d8f0c2913',
 'dcd21115-106b-

In [ ]:
# MMR


['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '50',
 '51',
 '52',
 '53',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '69',
 '70',
 '71',
 '72',
 '73',
 '74',
 '75',
 '76',
 '77',
 '78',
 '79',
 '80',
 '81',
 '82',
 '83',
 '84',
 '85',
 '86',
 '87',
 '88',
 '89',
 '90',
 '91',
 '92',
 '93',
 '94',
 '95',
 '96',
 '97',
 '98',
 '99',
 '100',
 '101',
 '102',
 '103',
 '104',
 '105',
 '106',
 '107',
 '108',
 '109',
 '110',
 '111',
 '112',
 '113',
 '114',
 '115',
 '116',
 '117',
 '118',
 '119',
 '120',
 '121',
 '122',
 '123',
 '124',
 '125',
 '126',
 '127',
 '128',
 '129',
 '130',
 '131',
 '132',
 '133',
 '134',
 '135',
 '136',
 '137',
 '138'

In [79]:
retriever = ml_vector_store.as_retriever(
    search_type = 'mmr',
        search_kwargs = {'k' : 100}
)

In [80]:
query = "why bagging reduces variance of the model as well as bias?"

In [81]:
relevant_docs = retriever.invoke(query)

content = []

for data in relevant_docs:
    content.append(data.page_content)

In [83]:
content = "".join(content)
print(len(content))

43204


In [101]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
load_dotenv()

def model():
    llm = HuggingFaceEndpoint(
        repo_id = 'openai/gpt-oss-120b',
        task = 'text-generation',
        max_new_tokens = 5000,
        temperature = 0.2
    )

    chatmodel = ChatHuggingFace(llm = llm)
    return chatmodel
model = model()

In [ ]:
# Multi qeury retriever : 

In [85]:
from langchain_classic.retrievers import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(retriever = retriever, llm = model)

In [86]:
query

'why bagging reduces variance of the model as well as bias?'

In [87]:
result = multi_query_retriever.invoke(query)

In [88]:
result
content = []
for data in result:
    content.append(data.page_content)
print("".join(content))

If the problem is classification, the final prediction is decided by the
majority vote of all models. For example, if 10 classifiers vote SPAM and 6
models vote NOT SPAM, the final prediction is SPAM.
If the problem is regression, the final prediction is the average of all
models’ predictions.
Bagging generally improves unstable methods, such as neural networks,
classification and regression trees, and subset selection in linear regression.
However, it can mildly degrade the performance of stable methods such as
k-nearest neighbors .
Figure 5-7. Bagging illustration by Sirakorn
A random forest is an example of bagging. A random forest is a collection
of decision trees constructed by both bagging and feature randomness,
11in many cases. This is especially true when data collection is expensive or
difficult and you have to rely on the data collected by someone else. As a
result, inputs in production are often noisy compared to inputs used in
development
. The model that performs best on 

In [98]:
# Contexual Compression Retrievers

from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv
load_dotenv()

from langchain_classic.retrievers import ContextualCompressionRetriever

In [104]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-3B-Instruct",  # Hugging Face model repo
    task = "text-generation",
    temperature = 0,
    max_new_tokens= 1000,
)

model = ChatHuggingFace(llm = llm)

In [100]:
model.invoke("what is machine learing?")

AIMessage(content='Machine learning (ML) is a subset of artificial intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. The goal of machine learning is to enable computers to automatically improve their performance on a task by learning from experience, without being explicitly programmed.\n\nMachine learning involves several key steps:\n\n1. **Data collection**: Gathering data relevant to the problem you want to solve.\n2. **Data preprocessing**: Cleaning, transforming, and preparing the data for use in the machine learning algorithm.\n3. **Model training**: Using the preprocessed data to train a machine learning model.\n4. **Model evaluation**: Assessing the performance of the trained model on a test dataset.\n5. **Model deployment**: Deploying the trained model in a production environment.\n\nMachine learning algorithms can be broadly classified into three categories:\n\n1. **Supervised learning

In [105]:
base_retriever = retriever
compressor = LLMChainExtractor.from_llm(model)
ccr = ContextualCompressionRetriever(base_retriever = base_retriever, 
                                     base_compressor = compressor) 

In [106]:
result = ccr.invoke(query)

In [107]:
result
content = []
for data in result:
    content.append(data.page_content)
print("".join(content))

The relevant part of the context is:

Bagging
Bagging, shortened from bootstrap aggregating, is designed to improve
both the training stability
 and accuracy of ML algorithms. It reduces
variance and helps to avoid overfitting.
Given a dataset, instead of training one classifier on the entire dataset, you
sample with replacement to create different datasets, called bootstraps, and
train a classification or regression model on each of these bootstraps.
Sampling with replacement ensures that each bootstrap is independent from
its peers. Figure 5-7 shows an illustration of bagging.


In [108]:
!pip install rank_bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever


# Option A: Initialize directly from a list of strings
bm25_retriever = BM25Retriever.from_documents(chunks)

# Option B: Initialize from standard LangChain Document objects
# docs = [Document(page_content=t) for t in texts]
# bm25_retriever = BM25Retriever.from_documents(docs)

# # Adjust the number of documents to return
bm25_retriever.k = 10

# Invoke the retriever
results = bm25_retriever.invoke(query)
for doc in results:
    print(doc.page_content)


Only 1 is correct
(0.3 * 0.3 * 0.7) * 3 = 0.189 Wrong
 
            
None is correct
0.3 * 0.3 * 0.3 = 0.027 Wrong
 
          
This calculation only holds if the classifiers in an ensemble are
uncorrelated. If all classifiers are perfectly correlated — all three of them
make the same prediction for every email — the ensemble will have the
same accuracy as each individual classifier. When creating an ensemble, the
less correlation there is among base learners, the better the ensemble will
be. Therefore, it’s common to choose very different types of models for an
ensemble. For example, you might create an ensemble that consists of one
transformer model, one recurrent neural network, and one gradient boosted
tree.
There are three ways to create an ensemble: bagging to reduce variance,
boosting to reduce bias, and stacking to help with generalization. Other than
to help boost performance, according to several survey papers, ensemble
methods such as boosting and bagging, together with resa